# WTI COT MM Nowcasting — Kalman Filter — 02 Baseline Multivariate KF

We implement and compare two **feature-free** baseline state space models:

| Model | State | Params |
|---|---|---|
| **A — Local Level (diag)** | Random walk, diagonal noise | 6 |
| **B — Local Level (full cov)** | Random walk, full covariance | 12 |
| **C — Local Linear Trend** | Random walk + slope, diagonal | 12 |

All models are estimated via **MLE** using `statsmodels.tsa.statespace.MLEModel`.  
Evaluation uses a **walk-forward (expanding window)** scheme to produce 1-step-ahead nowcasts.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../../../')

In [ ]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import statsmodels.api as sm
from statsmodels.tsa.statespace.mlemodel import MLEModel

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
from src.utils.io.read import PreprocessedDataReader
from src.preprocessing.base import FutureTicker
from src.settings import Settings

pdr = PreprocessedDataReader(Settings.historical.paths.PREPROCESSED_DATA_PATH)
dataset = pdr.read_dataset(ticker=FutureTicker.WTI)
dataset['tradeDate'] = pd.to_datetime(dataset['tradeDate'])
dataset.sort_values('tradeDate', inplace=True)
dataset.reset_index(drop=True, inplace=True)

print(f'Shape: {dataset.shape}')
print(f'Date range: {dataset["tradeDate"].min().date()} → {dataset["tradeDate"].max().date()}')

In [ ]:
# Load config saved in notebook 01
CONFIG_PATH = pathlib.Path('../../../cache/output/wti/mm/kf_config.json')
with open(CONFIG_PATH) as f:
    kf_config = json.load(f)

RESPONSES_RAW = kf_config['responses_raw']
RESPONSES_OI  = kf_config['responses_oi']
FEATURES       = kf_config['selected_features']

print('Responses (raw):', RESPONSES_RAW)
print('Responses (OI) :', RESPONSES_OI)

---
## 1. Data Preparation

We model the **raw position changes** (Net, Long, Short) as the multivariate observation vector `y_t ∈ ℝ³`.  
We drop any row with a NaN in any response to keep the observation matrix complete.

In [ ]:
# Build clean observation matrix
df = dataset[['tradeDate'] + RESPONSES_RAW].dropna().reset_index(drop=True)

dates = df['tradeDate'].values
Y = df[RESPONSES_RAW].values.astype(float)  # (T, 3)

print(f'Observation matrix Y: {Y.shape}  ({df["tradeDate"].min().date()} → {df["tradeDate"].max().date()})')
print(f'Response labels: Net, Long, Short')

# Quick sanity check
pd.DataFrame(Y, columns=['Net', 'Long', 'Short']).describe().round(2)

In [ ]:
# Standardise for numerical stability during MLE
# We store scale so predictions can be back-transformed
Y_mean = Y.mean(axis=0)
Y_std  = Y.std(axis=0)
Y_scaled = (Y - Y_mean) / Y_std

print(f'Mean per response : {Y_mean.round(1)}')
print(f'Std  per response : {Y_std.round(1)}')

---
## 2. State Space Models

### Notation (statsmodels convention)

```
Observation:  y_t  = Z α_t + ε_t,   ε_t ~ N(0, H)
State:       α_{t+1} = T α_t + R η_t,  η_t ~ N(0, Q)
```

**Local Level (random walk + noise):**
- `Z = I₃`, `T = I₃`, `R = I₃`
- Parameters: `H` (obs noise), `Q` (state noise)

**Local Linear Trend:**
- State `α_t = [level_t (3), slope_t (3)]` → 6-dim
- `Z = [I₃ | 0₃]`
- `T = [[I₃, I₃], [0₃, I₃]]`
- Parameters: `H`, `Q_level`, `Q_slope`

In [ ]:
class MultivariateLocalLevel(MLEModel):
    """
    Multivariate local level (random walk plus noise) model.

    Observation:  y_t  = α_t + ε_t   ε_t ~ N(0, H)
    State:       α_t+1 = α_t + η_t   η_t ~ N(0, Q)

    Parameters (log-scale for positivity):
        log_h_diag  (k,)   — log of observation std devs  (diagonal H)
        log_q_diag  (k,)   — log of state std devs        (diagonal Q)
        h_offdiag   (k*(k-1)/2,)  — lower-triangular obs cov elements (full variant)
    """

    def __init__(self, endog, full_cov=False):
        k = endog.shape[1]
        self.k_endog_  = k
        self.full_cov  = full_cov
        super().__init__(endog, k_states=k, k_posdef=k)

        # Fixed system matrices
        self['design']     = np.eye(k)          # Z
        self['transition'] = np.eye(k)          # T
        self['selection']  = np.eye(k)          # R

        self.initialize_approximate_diffuse()

    # ------------------------------------------------------------------ #
    @property
    def param_names(self):
        k = self.k_endog_
        names = [f'log_h_{i}' for i in range(k)]   # obs noise
        names += [f'log_q_{i}' for i in range(k)]  # state noise
        if self.full_cov:
            # lower-triangular off-diagonal elements of H
            for i in range(1, k):
                for j in range(i):
                    names.append(f'h_cov_{i}{j}')
        return names

    @property
    def start_params(self):
        k = self.k_endog_
        # Initialise at log(sample std) for each series
        log_std = np.log(np.std(self.endog, axis=0) + 1e-6)
        p = np.concatenate([log_std, log_std - 1.0])
        if self.full_cov:
            n_off = k * (k - 1) // 2
            p = np.concatenate([p, np.zeros(n_off)])
        return p

    # ------------------------------------------------------------------ #
    def update(self, params, **kwargs):
        params = super().update(params, **kwargs)
        k = self.k_endog_

        h_std = np.exp(params[:k])
        q_std = np.exp(params[k:2*k])

        if self.full_cov:
            # Build lower-triangular Cholesky factor for H
            L = np.diag(h_std)
            off = params[2*k:]
            idx = 0
            for i in range(1, k):
                for j in range(i):
                    L[i, j] = off[idx]
                    idx += 1
            H = L @ L.T
        else:
            H = np.diag(h_std ** 2)

        Q = np.diag(q_std ** 2)

        self['obs_cov']   = H
        self['state_cov'] = Q

In [ ]:
class MultivariateLocalLinearTrend(MLEModel):
    """
    Multivariate local linear trend model (diagonal covariances).

    State α_t = [level_t (k), slope_t (k)]  →  2k-dimensional

    Observation:  y_t = [I_k | 0_k] α_t + ε_t
    State:        level_{t+1} = level_t + slope_t + η1_t
                  slope_{t+1} = slope_t             + η2_t

    Parameters (log-scale):
        log_h_diag  (k,)  — obs noise std devs
        log_q1_diag (k,)  — level noise std devs
        log_q2_diag (k,)  — slope noise std devs
    """

    def __init__(self, endog):
        k = endog.shape[1]
        self.k_endog_ = k
        # 2k states, k posdef (we use block-diagonal Q internally)
        super().__init__(endog, k_states=2*k, k_posdef=2*k)

        # Design: Z = [I_k | 0_k]
        Z = np.zeros((k, 2*k))
        Z[:, :k] = np.eye(k)
        self['design'] = Z

        # Transition: T = [[I, I], [0, I]]
        T = np.eye(2*k)
        T[:k, k:] = np.eye(k)
        self['transition'] = T

        # Selection: R = I_{2k}
        self['selection'] = np.eye(2*k)

        self.initialize_approximate_diffuse()

    # ------------------------------------------------------------------ #
    @property
    def param_names(self):
        k = self.k_endog_
        names  = [f'log_h_{i}'  for i in range(k)]
        names += [f'log_q1_{i}' for i in range(k)]
        names += [f'log_q2_{i}' for i in range(k)]
        return names

    @property
    def start_params(self):
        k = self.k_endog_
        log_std = np.log(np.std(self.endog, axis=0) + 1e-6)
        return np.concatenate([log_std, log_std - 1.0, log_std - 3.0])

    # ------------------------------------------------------------------ #
    def update(self, params, **kwargs):
        params = super().update(params, **kwargs)
        k = self.k_endog_

        h_std  = np.exp(params[:k])
        q1_std = np.exp(params[k:2*k])
        q2_std = np.exp(params[2*k:])

        self['obs_cov']   = np.diag(h_std ** 2)

        Q = np.zeros((2*k, 2*k))
        Q[:k, :k] = np.diag(q1_std ** 2)   # level noise
        Q[k:, k:] = np.diag(q2_std ** 2)   # slope noise
        self['state_cov'] = Q

---
## 3. Full-Sample Fit

Fit all three models on the entire history to inspect filter/smoother quality before the walk-forward evaluation.

In [ ]:
def fit_model(model_cls, endog, **model_kwargs):
    mod = model_cls(endog, **model_kwargs)
    res = mod.fit(disp=False, method='lbfgs', maxiter=500)
    return mod, res

print('Fitting Model A — Local Level (diagonal) ...')
mod_A, res_A = fit_model(MultivariateLocalLevel, Y_scaled, full_cov=False)
print(f'  Log-likelihood: {res_A.llf:.2f}   AIC: {res_A.aic:.2f}   BIC: {res_A.bic:.2f}')

print('Fitting Model B — Local Level (full cov) ...')
mod_B, res_B = fit_model(MultivariateLocalLevel, Y_scaled, full_cov=True)
print(f'  Log-likelihood: {res_B.llf:.2f}   AIC: {res_B.aic:.2f}   BIC: {res_B.bic:.2f}')

print('Fitting Model C — Local Linear Trend ...')
mod_C, res_C = fit_model(MultivariateLocalLinearTrend, Y_scaled)
print(f'  Log-likelihood: {res_C.llf:.2f}   AIC: {res_C.aic:.2f}   BIC: {res_C.bic:.2f}')

In [ ]:
# In-sample information criteria summary
results_summary = pd.DataFrame({
    'Model': ['A: Local Level (diag)', 'B: Local Level (full cov)', 'C: Local Linear Trend'],
    'n_params': [len(res_A.params), len(res_B.params), len(res_C.params)],
    'LogLik': [res_A.llf, res_B.llf, res_C.llf],
    'AIC': [res_A.aic, res_B.aic, res_C.aic],
    'BIC': [res_A.bic, res_B.bic, res_C.bic],
}).set_index('Model').round(2)
results_summary

### 3.1 Filter vs Smoother Plots

- **Filter**: $E[\alpha_t \mid y_1, \ldots, y_t]$ — real-time estimate, available for nowcasting  
- **Smoother**: $E[\alpha_t \mid y_1, \ldots, y_T]$ — retrospective estimate, uses all data

In [ ]:
def plot_filter_smoother(result, Y_scaled, Y_std, Y_mean, dates, title, k=3, n_state_per_response=1):
    """
    Plot filtered and smoothed state estimates vs observations (back-transformed).
    Works for Local Level (k states = k_endog) and LLT (first k states = level).
    """
    labels = ['Net', 'Long', 'Short']
    filtered  = result.filtered_state[:k].T * Y_std + Y_mean    # (T, k)
    smoothed  = result.smoothed_state[:k].T * Y_std + Y_mean    # (T, k)
    observed  = Y  # already in original scale

    fig, axes = plt.subplots(k, 1, figsize=(14, 3.5 * k), sharex=True)
    for i, (ax, label) in enumerate(zip(axes, labels)):
        ax.bar(dates, observed[:, i], color='lightsteelblue', width=5, alpha=0.6, label='Observed')
        ax.plot(dates, filtered[:, i],  color='darkorange', linewidth=1.2, label='Filtered')
        ax.plot(dates, smoothed[:, i],  color='darkblue',   linewidth=1.2, linestyle='--', label='Smoothed')
        ax.axhline(0, color='grey', linestyle=':', linewidth=0.8)
        ax.set_ylabel('Contracts')
        ax.set_title(f'{label} Position Change')
        ax.legend(loc='upper right', fontsize=8)
    axes[-1].set_xlabel('Date')
    fig.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

plot_filter_smoother(res_A, Y_scaled, Y_std, Y_mean, dates,
                     'Model A — Local Level (diagonal): Filter vs Smoother')
plot_filter_smoother(res_B, Y_scaled, Y_std, Y_mean, dates,
                     'Model B — Local Level (full cov): Filter vs Smoother')
plot_filter_smoother(res_C, Y_scaled, Y_std, Y_mean, dates,
                     'Model C — Local Linear Trend: Filter vs Smoother')

In [ ]:
# Standardised residuals diagnostics for Model A
fig = res_A.plot_diagnostics(variable=0, figsize=(14, 6))
plt.suptitle('Model A — Standardised Residuals Diagnostics (Net)', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Walk-Forward (Expanding Window) Evaluation

**Scheme:**  
- Minimum training size: `MIN_TRAIN` observations  
- At each step `t ≥ MIN_TRAIN`: fit model on `Y[:t]`, predict `y_{t+1}` using the filtered state  
- Collect 1-step-ahead predictions across the out-of-sample window

**Note:** We refit MLE parameters every `REFIT_EVERY` steps to balance accuracy and speed.

In [ ]:
MIN_TRAIN   = 200   # ~4 years of weekly data
REFIT_EVERY = 26    # refit MLE params every 26 weeks (6 months)

print(f'Walk-forward: training starts at {MIN_TRAIN} obs, refitting every {REFIT_EVERY} steps')
print(f'Out-of-sample window: {len(Y) - MIN_TRAIN} observations  '
      f'({dates[MIN_TRAIN].astype("datetime64[D]")!s} → {dates[-1].astype("datetime64[D]")!s})')

In [ ]:
def walk_forward(model_cls, Y_scaled, Y_mean, Y_std, min_train, refit_every, **model_kwargs):
    """
    Walk-forward 1-step-ahead evaluation.
    Returns a DataFrame of predictions vs actuals (original scale).
    """
    T = len(Y_scaled)
    preds = []   # list of (t, pred_net, pred_long, pred_short)
    current_params = None

    for t in range(min_train, T - 1):
        Y_train = Y_scaled[:t]

        # Refit MLE params periodically
        if current_params is None or (t - min_train) % refit_every == 0:
            mod = model_cls(Y_train, **model_kwargs)
            try:
                res = mod.fit(disp=False, method='lbfgs', maxiter=300,
                              start_params=current_params)
                current_params = res.params
            except Exception:
                pass  # keep previous params if optimisation fails
        else:
            # Use fixed params, just run the filter
            mod = model_cls(Y_train, **model_kwargs)
            res = mod.filter(current_params)

        # 1-step-ahead forecast = filtered state at t (last observation)
        # For Local Level: E[y_{t+1}] = E[alpha_{t+1}|y_{1..t}] = T * alpha_t|t
        filtered_state_last = res.filtered_state[:, -1]   # (k_states,)
        T_mat = mod['transition']
        Z_mat = mod['design']
        forecast_state = T_mat @ filtered_state_last       # predicted state
        forecast_obs   = Z_mat @ forecast_state            # predicted obs (scaled)

        # Back-transform
        pred_orig = forecast_obs[:3] * Y_std + Y_mean
        preds.append(pred_orig)

    preds = np.array(preds)   # (n_oos, 3)
    actuals = Y[min_train + 1:]  # observed y_{t+1}, original scale

    return pd.DataFrame({
        'date': dates[min_train + 1:],
        'pred_net':   preds[:, 0],
        'pred_long':  preds[:, 1],
        'pred_short': preds[:, 2],
        'actual_net':   actuals[:, 0],
        'actual_long':  actuals[:, 1],
        'actual_short': actuals[:, 2],
    })

print('Running walk-forward for Model A (Local Level diagonal)...')
wf_A = walk_forward(MultivariateLocalLevel, Y_scaled, Y_mean, Y_std,
                    MIN_TRAIN, REFIT_EVERY, full_cov=False)

print('Running walk-forward for Model B (Local Level full cov)...')
wf_B = walk_forward(MultivariateLocalLevel, Y_scaled, Y_mean, Y_std,
                    MIN_TRAIN, REFIT_EVERY, full_cov=True)

print('Running walk-forward for Model C (Local Linear Trend)...')
wf_C = walk_forward(MultivariateLocalLinearTrend, Y_scaled, Y_mean, Y_std,
                    MIN_TRAIN, REFIT_EVERY)

print(f'Done. OOS window: {len(wf_A)} observations.')

In [ ]:
def compute_metrics(wf_df):
    """Spearman ρ, RMSE, and directional accuracy per response."""
    responses = ['net', 'long', 'short']
    rows = []
    for r in responses:
        pred   = wf_df[f'pred_{r}'].values
        actual = wf_df[f'actual_{r}'].values
        mask   = ~(np.isnan(pred) | np.isnan(actual))
        rho, _ = stats.spearmanr(pred[mask], actual[mask])
        rmse   = np.sqrt(np.mean((pred[mask] - actual[mask]) ** 2))
        dir_acc = np.mean(np.sign(pred[mask]) == np.sign(actual[mask]))
        rows.append({'Response': r.capitalize(), 'Spearman ρ': rho,
                     'RMSE': rmse, 'Dir. Accuracy': dir_acc})
    return pd.DataFrame(rows).set_index('Response')

metrics_A = compute_metrics(wf_A)
metrics_B = compute_metrics(wf_B)
metrics_C = compute_metrics(wf_C)

print('Model A — Local Level (diagonal):')
print(metrics_A.round(4), '\n')
print('Model B — Local Level (full cov):')
print(metrics_B.round(4), '\n')
print('Model C — Local Linear Trend:')
print(metrics_C.round(4))

In [ ]:
# Combined metrics table
for col, label in [('Spearman ρ', 'Spearman ρ'), ('RMSE', 'RMSE'), ('Dir. Accuracy', 'Dir. Accuracy')]:
    tab = pd.concat([
        metrics_A[[col]].rename(columns={col: 'A: LL-diag'}),
        metrics_B[[col]].rename(columns={col: 'B: LL-full'}),
        metrics_C[[col]].rename(columns={col: 'C: LLT'}),
    ], axis=1)
    print(f'\n--- {label} ---')
    print(tab.round(4))

In [ ]:
# Walk-forward prediction plots
def plot_wf_predictions(wf_df, model_name):
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    for ax, r, color in zip(axes, ['net', 'long', 'short'], ['steelblue', 'green', 'tomato']):
        ax.bar(wf_df['date'], wf_df[f'actual_{r}'],
               color='lightgrey', width=5, alpha=0.8, label='Actual')
        ax.plot(wf_df['date'], wf_df[f'pred_{r}'],
                color=color, linewidth=1.2, label='1-step forecast')
        ax.axhline(0, color='grey', linestyle=':', linewidth=0.8)
        rho, _ = stats.spearmanr(
            wf_df[f'pred_{r}'].dropna(), wf_df[f'actual_{r}'].dropna()
        )
        ax.set_title(f'{r.capitalize()} Δ  (Spearman ρ = {rho:.3f})')
        ax.set_ylabel('Contracts')
        ax.legend(loc='upper right', fontsize=8)
    axes[-1].set_xlabel('Date')
    fig.suptitle(f'{model_name} — Walk-forward 1-step Nowcasts', fontsize=12)
    plt.tight_layout()
    plt.show()

plot_wf_predictions(wf_A, 'Model A: Local Level (diagonal)')
plot_wf_predictions(wf_B, 'Model B: Local Level (full cov)')
plot_wf_predictions(wf_C, 'Model C: Local Linear Trend')

In [ ]:
# Scatter: predicted vs actual (Net) for each model
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (wf, name) in zip(axes, [
    (wf_A, 'A: Local Level (diag)'),
    (wf_B, 'B: Local Level (full)'),
    (wf_C, 'C: Local Linear Trend'),
]):
    pred   = wf['pred_net'].values
    actual = wf['actual_net'].values
    mask   = ~(np.isnan(pred) | np.isnan(actual))
    rho, _ = stats.spearmanr(pred[mask], actual[mask])
    ax.scatter(actual[mask], pred[mask], alpha=0.3, s=10, color='steelblue')
    lim = max(np.abs(actual[mask]).max(), np.abs(pred[mask]).max())
    ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1)
    ax.set_title(f'{name}\nSpearman ρ = {rho:.3f}')
    ax.set_xlabel('Actual Net Δ')
    ax.set_ylabel('Predicted Net Δ')
plt.suptitle('Predicted vs Actual — Net Position Change (OOS)', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Estimated Parameters (Full-Sample Fit)

Inspect the fitted noise ratios — the **signal-to-noise ratio** (Q/H) tells us how much of the variation is attributed to the latent state vs observation noise.

In [ ]:
labels = ['Net', 'Long', 'Short']

for model_name, res, mod in [
    ('A: Local Level (diag)', res_A, mod_A),
    ('C: Local Linear Trend', res_C, mod_C),
]:
    print(f'\n=== {model_name} ===')
    H = mod['obs_cov']
    Q = mod['state_cov']
    print('  Obs noise std (H diagonal)  :', np.sqrt(np.diag(H)).round(4), ' (scaled)')
    print('  State noise std (Q diagonal):', np.sqrt(np.diag(Q)[:3]).round(4), ' (scaled)')
    snr = np.diag(Q)[:3] / (np.diag(H) + 1e-10)
    print('  Signal-to-noise (Q/H)       :', snr.round(4))

---
## 6. Summary

| Model | AIC | OOS Spearman Net | OOS Spearman Long | OOS Spearman Short |
|---|---|---|---|---|
| A: Local Level (diag) | — | — | — | — |
| B: Local Level (full) | — | — | — | — |
| C: Local Linear Trend | — | — | — | — |

*(Fill in from cells above)*

**Observations:**
- The baseline KF (without any features) already captures some of the positioning dynamic through the latent state
- The signal-to-noise ratio shows how much of the weekly change is predictable from the state vs noise
- In notebook 03 we add the 5 exogenous features to the observation equation to improve nowcast quality

### Design choice for Notebook 03
Carry forward the **best-performing baseline** as the state component, and add features as:
1. Fixed-coefficient regressors: `y_t = Z α_t + X_t β + ε_t`
2. Time-varying-coefficient regressors: `β_t` evolves as an additional state

In [ ]:
# Save baseline walk-forward results for comparison in notebook 04
OUT_DIR = pathlib.Path('../../../cache/output/wti/mm')

wf_A.to_csv(OUT_DIR / 'kf_wf_baseline_A.csv', index=False)
wf_B.to_csv(OUT_DIR / 'kf_wf_baseline_B.csv', index=False)
wf_C.to_csv(OUT_DIR / 'kf_wf_baseline_C.csv', index=False)

print('Walk-forward results saved.')